# 光伏场站中位数曲线与单调时间配准

这个 Notebook 用于回答两个问题：

1. 不同场站在装机容量/幅值归一化后，中位数日曲线是否仍存在稳定的形状差异？
2. 只改变时间坐标的单调非线性配准，能消除多少差异？

数据文件格式：`station=xxx.parquet`。默认认为 `timestamp_win` 对应 `observe_power` 的最后一个元素；每行只取这个元素重建15分钟功率序列，避免7天滑动窗口对同一时刻反复计数。`observe_power_future` 不会被使用，因此不会把未来功率泄漏到配准模板中。

配准定义为：在公共时间 $\tau$ 上，通过单调映射 $t=\psi_s(\tau)$ 读取场站曲线，得到 $q_s(\tau)=p_s(\psi_s(\tau))$。优化只拉伸或压缩时间轴，不主动改变功率幅值；重新采样时会产生正常的插值变化。

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy.optimize import minimize

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


## 1. 配置

`NORMALIZATION="capacity"` 最符合你的正式实验，但需要在 `CAPACITY_BY_STATION` 中填写每个站的装机容量。为了让 Notebook 拿到当前三列数据就能直接运行，默认先使用 `p95`，它只适合观察形状，不应替代正式实验里的装机容量归一化。

如果 `timestamp_win` 表示未来起点，而最后一个历史点实际是它前15分钟，把 `HISTORY_LAST_OFFSET` 改成 `"-15min"`。

中文字体会自动从常见字体中选择。如果服务器没有安装中文字体，把一个 `.ttf`、`.otf` 或 `.ttc` 字体文件的绝对路径填入 `CHINESE_FONT_PATH`，例如 Noto Sans CJK 或思源黑体。

In [ ]:
# ===== 只需要优先修改这里 =====
DATA_DIR = Path("/data/hjs/your_parquet_directory")
FILE_GLOB = "station=*.parquet"
CHINESE_FONT_PATH = None  # 例如："/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

TIME_COL = "timestamp_win"
POWER_HISTORY_COL = "observe_power"
STATION_COL = "station"
HISTORY_LAST_OFFSET = "0min"       # 常见的另一种定义是 "-15min"

# 可选值："capacity"、"p95"、"peak"、"none"
NORMALIZATION = "p95"
CAPACITY_BY_STATION = {
    # "001": 465.0,
    # "002": 520.0,
}

# None：以所有场站曲线的中位数为公共模板；也可以填某个 station 字符串作为固定参考站。
REFERENCE_STATION = None

# 数据质量与配准强度
MIN_OBSERVATIONS_PER_SLOT = 10
N_WARP_KNOTS = 9
MIN_KNOT_GAP = 0.025
IDENTITY_PENALTY = 0.015
SMOOTHNESS_PENALTY = 0.010
DAYLIGHT_THRESHOLD = 0.02
N_TEMPLATE_ITERATIONS = 3


def configure_chinese_font(font_path=None):
    """配置 Matplotlib 中文字体，返回实际使用的字体名称。"""
    common_paths = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc",
        "C:/Windows/Fonts/simhei.ttf",
    ]

    selected_path = Path(font_path).expanduser() if font_path else None
    if selected_path is not None and not selected_path.exists():
        raise FileNotFoundError(f"CHINESE_FONT_PATH 不存在：{selected_path}")
    if selected_path is None:
        selected_path = next((Path(p) for p in common_paths if Path(p).exists()), None)

    if selected_path is not None:
        font_manager.fontManager.addfont(str(selected_path))
        font_name = font_manager.FontProperties(fname=str(selected_path)).get_name()
    else:
        candidates = [
            "Noto Sans CJK SC",
            "Noto Sans CJK JP",
            "Source Han Sans SC",
            "Microsoft YaHei",
            "SimHei",
            "PingFang SC",
            "WenQuanYi Micro Hei",
            "Arial Unicode MS",
        ]
        installed = {font.name for font in font_manager.fontManager.ttflist}
        font_name = next((name for name in candidates if name in installed), None)

    if font_name is None:
        warnings.warn(
            "没有检测到中文字体。请安装 Noto Sans CJK/思源黑体，"
            "或在 CHINESE_FONT_PATH 中填写字体文件的绝对路径。"
        )
        return None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = [font_name, "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    print(f"Matplotlib 中文字体：{font_name}")
    return font_name


CHINESE_FONT_NAME = configure_chinese_font(CHINESE_FONT_PATH)


## 2. 读取 parquet 并重建15分钟序列

这里不展开全部7天数组，而是取每行 `observe_power[-1]`。如果文件确实每15分钟一行，这正好为每个物理时刻保留一个观测值，内存占用也远小于展开滑动窗口。

In [ ]:
def last_history_value(x):
    """严格返回历史数组最后一个值，避免把更早的值错误地放到当前时间戳。"""
    if x is None:
        return np.nan
    arr = np.asarray(x, dtype=float).reshape(-1)
    if arr.size == 0:
        return np.nan
    return float(arr[-1]) if np.isfinite(arr[-1]) else np.nan


def station_name_from_file(path, frame):
    if STATION_COL in frame.columns:
        values = frame[STATION_COL].dropna()
        if not values.empty:
            return str(values.iloc[0])
    return path.stem.split("station=", 1)[-1]


def load_station_series(path):
    frame = pd.read_parquet(path, columns=[TIME_COL, POWER_HISTORY_COL, STATION_COL])
    station = station_name_from_file(path, frame)
    timestamps = pd.to_datetime(frame[TIME_COL], errors="coerce") + pd.Timedelta(HISTORY_LAST_OFFSET)
    power = frame[POWER_HISTORY_COL].map(last_history_value).astype(float)

    series = pd.DataFrame({"timestamp": timestamps, "power": power}).dropna()
    # 若同一时间戳有重复行，用中位数合并。
    series = series.groupby("timestamp", as_index=False)["power"].median().sort_values("timestamp")
    series["station"] = station
    return series


files = sorted(DATA_DIR.glob(FILE_GLOB))
if not files:
    raise FileNotFoundError(f"在 {DATA_DIR} 下没有找到 {FILE_GLOB}")

station_series = {}
for path in files:
    series = load_station_series(path)
    station = str(series["station"].iloc[0])
    if station in station_series:
        raise ValueError(f"station={station} 出现在多个文件中，请检查命名或 station 列")
    station_series[station] = series

summary = pd.DataFrame([
    {
        "station": station,
        "n_timestamps": len(frame),
        "start": frame["timestamp"].min(),
        "end": frame["timestamp"].max(),
        "median_step_min": frame["timestamp"].diff().dt.total_seconds().median() / 60,
    }
    for station, frame in station_series.items()
]).sort_values("station").reset_index(drop=True)

print(f"读取到 {len(station_series)} 个场站")
display(summary)


## 3. 计算每个场站的中位数日曲线

先把所有时间戳映射到一天中的96个15分钟槽位，再对每个槽位跨天取中位数。缺失槽位使用相邻时刻线性插值；观测数不足的槽位会先置为缺失。

In [ ]:
SLOTS_PER_DAY = 96
slot_grid = np.arange(SLOTS_PER_DAY)
hour_grid = slot_grid / 4.0


def median_daily_curve(frame):
    ts = frame["timestamp"]
    slot = ts.dt.hour * 4 + ts.dt.minute // 15
    grouped = frame.assign(slot=slot).groupby("slot")["power"]
    med = grouped.median().reindex(slot_grid)
    count = grouped.count().reindex(slot_grid, fill_value=0)
    med[count < MIN_OBSERVATIONS_PER_SLOT] = np.nan
    med = med.interpolate(limit_direction="both")
    if med.isna().any():
        raise ValueError("中位数曲线仍包含 NaN；请降低 MIN_OBSERVATIONS_PER_SLOT 或检查数据覆盖")
    return med.to_numpy(dtype=float), count.to_numpy(dtype=int)


def normalize_curve(curve, station):
    curve = np.asarray(curve, dtype=float)
    if NORMALIZATION == "none":
        scale = 1.0
    elif NORMALIZATION == "capacity":
        if station not in CAPACITY_BY_STATION:
            raise KeyError(f"CAPACITY_BY_STATION 中缺少 station={station} 的装机容量")
        scale = float(CAPACITY_BY_STATION[station])
    elif NORMALIZATION == "p95":
        scale = float(np.nanpercentile(curve, 95))
    elif NORMALIZATION == "peak":
        scale = float(np.nanmax(curve))
    else:
        raise ValueError(f"未知 NORMALIZATION={NORMALIZATION}")
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f"station={station} 的归一化尺度无效：{scale}")
    return curve / scale, scale


curves = {}
counts = {}
scales = {}
for station, frame in station_series.items():
    raw_curve, slot_count = median_daily_curve(frame)
    curves[station], scales[station] = normalize_curve(raw_curve, station)
    counts[station] = slot_count

curves_df = pd.DataFrame(curves, index=slot_grid)
curves_df.index.name = "slot"
scale_table = pd.DataFrame({"station": list(scales), "normalization_scale": list(scales.values())})

print(f"归一化方式：{NORMALIZATION}")
display(scale_table.sort_values("station").reset_index(drop=True))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for station in curves_df.columns:
    ax.plot(hour_grid, curves_df[station], lw=1.7, alpha=0.80, label=station)
ax.set(title="各场站配准前的中位数日曲线", xlabel="原始时刻（小时）", ylabel=f"功率（{NORMALIZATION} 归一化）")
ax.set_xlim(0, 23.75)
ax.set_xticks(np.arange(0, 25, 2))
ax.legend(ncol=min(4, max(1, math.ceil(len(curves_df.columns) / 8))), fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. 最优单调非线性时间配准

用若干控制点表示 $t=\psi_s(\tau)$。优化目标由三部分组成：与公共模板的均方误差、偏离恒等映射的惩罚、时间变换不平滑的惩罚。控制点必须严格递增，因此不会颠倒时间顺序。

`IDENTITY_PENALTY` 越大，变换越保守；`SMOOTHNESS_PENALTY` 越大，局部拉伸越平滑。不要仅为了得到很低的配准误差而把这两个值设得过小。

In [ ]:
grid01 = np.linspace(0.0, 1.0, SLOTS_PER_DAY)
canonical_knots = np.linspace(0.0, 1.0, N_WARP_KNOTS)


def apply_warp(curve, source_knots):
    """在公共时间 tau 上，按 t=psi(tau) 从原曲线插值取值。"""
    source_position = np.interp(grid01, canonical_knots, source_knots)
    registered = np.interp(source_position, grid01, curve)
    return registered, source_position


def fit_monotone_warp(curve, template):
    x0 = canonical_knots[1:-1].copy()
    bounds = [(MIN_KNOT_GAP, 1.0 - MIN_KNOT_GAP)] * len(x0)

    def unpack(x):
        return np.r_[0.0, x, 1.0]

    def monotone_constraint(x):
        return np.diff(unpack(x)) - MIN_KNOT_GAP

    def objective(x):
        source_knots = unpack(x)
        registered, _ = apply_warp(curve, source_knots)
        daylight = (template > DAYLIGHT_THRESHOLD) | (registered > DAYLIGHT_THRESHOLD)
        if daylight.sum() < 8:
            daylight = np.ones_like(template, dtype=bool)
        fit_loss = np.mean((registered[daylight] - template[daylight]) ** 2)
        identity_loss = np.mean((source_knots - canonical_knots) ** 2)
        smoothness_loss = np.mean(np.diff(source_knots, n=2) ** 2)
        return fit_loss + IDENTITY_PENALTY * identity_loss + SMOOTHNESS_PENALTY * smoothness_loss

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints={"type": "ineq", "fun": monotone_constraint},
        options={"maxiter": 800, "ftol": 1e-11, "disp": False},
    )
    if not result.success:
        warnings.warn(f"配准优化未完全收敛：{result.message}")
    source_knots = unpack(result.x)
    registered, source_position = apply_warp(curve, source_knots)
    return registered, source_position, source_knots, result


def initial_template(curves_frame):
    if REFERENCE_STATION is not None:
        key = str(REFERENCE_STATION)
        if key not in curves_frame.columns:
            raise KeyError(f"REFERENCE_STATION={key} 不在场站列表中")
        return curves_frame[key].to_numpy(dtype=float)
    return np.nanmedian(curves_frame.to_numpy(dtype=float), axis=1)


template = initial_template(curves_df)
iterations = 1 if REFERENCE_STATION is not None else N_TEMPLATE_ITERATIONS

for iteration in range(iterations):
    aligned_for_update = {}
    for station in curves_df.columns:
        aligned_for_update[station] = fit_monotone_warp(curves_df[station].to_numpy(), template)[0]
    if REFERENCE_STATION is None:
        new_template = np.nanmedian(pd.DataFrame(aligned_for_update).to_numpy(), axis=1)
        delta = float(np.sqrt(np.mean((new_template - template) ** 2)))
        print(f"模板迭代 {iteration + 1}/{iterations}，变化 RMSE={delta:.6f}")
        template = new_template

# 对最终模板重新拟合并保留结果
registered = {}
warp_positions = {}
warp_knots = {}
optimizer_rows = []
for station in curves_df.columns:
    reg, positions, knots, result = fit_monotone_warp(curves_df[station].to_numpy(), template)
    registered[station] = reg
    warp_positions[station] = positions
    warp_knots[station] = knots
    optimizer_rows.append({"station": station, "success": result.success, "objective": result.fun, "iterations": result.nit})

registered_df = pd.DataFrame(registered, index=slot_grid)
warp_positions_df = pd.DataFrame(warp_positions, index=slot_grid)
optimizer_table = pd.DataFrame(optimizer_rows)
display(optimizer_table)


## 5. 配准前后对比

左图是原始中位数曲线，右图是重新映射到公共时间轴后的曲线。黑色虚线为最终公共模板。若右图仍存在稳定的纵向差异（削顶、凹陷、峰高不同），说明这些差异不能仅靠时间配准解决。

In [ ]:
colors = plt.cm.tab20(np.linspace(0, 1, max(1, len(curves_df.columns))))
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for color, station in zip(colors, curves_df.columns):
    axes[0].plot(hour_grid, curves_df[station], color=color, lw=1.5, alpha=0.78, label=station)
    axes[1].plot(hour_grid, registered_df[station], color=color, lw=1.5, alpha=0.78, label=station)

axes[0].plot(hour_grid, template, "k--", lw=2.4, label="最终模板")
axes[1].plot(hour_grid, template, "k--", lw=2.4, label="最终模板")
axes[0].set_title("配准前：原始时间轴")
axes[1].set_title("配准后：公共时间轴")
for ax in axes:
    ax.set_xlabel("时刻（小时）")
    ax.set_xlim(0, 23.75)
    ax.set_xticks(np.arange(0, 25, 2))
axes[0].set_ylabel(f"功率（{NORMALIZATION} 归一化）")
axes[1].legend(ncol=min(3, max(1, math.ceil((len(curves_df.columns) + 1) / 8))), fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# 时间映射图：横轴是公共时间 tau，纵轴是从原曲线读取的物理时间 t=psi(tau)。
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(hour_grid, hour_grid, "k--", lw=1.8, label="恒等映射")
for color, station in zip(colors, curves_df.columns):
    ax.plot(hour_grid, warp_positions_df[station] * 23.75, color=color, lw=1.6, alpha=0.85, label=station)
ax.set(
    title="各场站的最优单调时间映射",
    xlabel="公共时间 τ（小时）",
    ylabel="原始物理时间 t = ψ(τ)（小时）",
    xlim=(0, 23.75),
    ylim=(0, 23.75),
)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_yticks(np.arange(0, 25, 3))
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()


In [ ]:
def daylight_rmse(curve, reference):
    mask = (curve > DAYLIGHT_THRESHOLD) | (reference > DAYLIGHT_THRESHOLD)
    if mask.sum() < 8:
        mask = np.ones_like(reference, dtype=bool)
    return float(np.sqrt(np.mean((curve[mask] - reference[mask]) ** 2)))


metric_rows = []
for station in curves_df.columns:
    before = daylight_rmse(curves_df[station].to_numpy(), template)
    after = daylight_rmse(registered_df[station].to_numpy(), template)
    metric_rows.append({
        "station": station,
        "rmse_before": before,
        "rmse_after": after,
        "rmse_reduction_pct": 100.0 * (before - after) / before if before > 0 else np.nan,
        "max_time_displacement_min": float(np.max(np.abs(warp_positions_df[station].to_numpy() - grid01)) * 23.75 * 60),
    })

metrics = pd.DataFrame(metric_rows).sort_values("rmse_reduction_pct", ascending=False).reset_index(drop=True)
print("配准前后相对于最终模板的白天 RMSE：")
display(metrics.style.format({
    "rmse_before": "{:.5f}",
    "rmse_after": "{:.5f}",
    "rmse_reduction_pct": "{:.2f}%",
    "max_time_displacement_min": "{:.1f}",
}))


## 6. 单站细看（可选）

修改 `STATION_TO_INSPECT`，可以同时查看该站配准前后曲线和时间映射。重点观察：配准后仍然存在的峰高差、削顶、局部凹陷和多峰结构。

In [ ]:
STATION_TO_INSPECT = str(curves_df.columns[0])

if STATION_TO_INSPECT not in curves_df.columns:
    raise KeyError(f"找不到 station={STATION_TO_INSPECT}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(hour_grid, curves_df[STATION_TO_INSPECT], lw=2, label="配准前")
axes[0].plot(hour_grid, registered_df[STATION_TO_INSPECT], lw=2, label="配准后")
axes[0].plot(hour_grid, template, "k--", lw=2, label="公共模板")
axes[0].set(title=f"station={STATION_TO_INSPECT} 曲线对比", xlabel="时刻（小时）", ylabel="归一化功率")
axes[0].legend()

axes[1].plot(hour_grid, warp_positions_df[STATION_TO_INSPECT] * 23.75, lw=2, label="最优映射")
axes[1].plot(hour_grid, hour_grid, "k--", lw=1.5, label="恒等映射")
axes[1].set(title="时间映射", xlabel="公共时间 τ（小时）", ylabel="原始时间 t（小时）")
axes[1].legend()
plt.tight_layout()
plt.show()


## 结果解释

- 配准后曲线明显收拢，并且时间映射平滑、偏移量合理：说明跨站差异中确实存在可利用的时间相位/宽度差异。
- RMSE 降低很多，但某些站仍有稳定纵向残差：时间配准只能解决部分 OOD 差异。
- 时间映射出现剧烈弯折或最大位移过大：优化可能在用不合理的时间扭曲拟合幅值差异，应增大两个惩罚项或减少控制点。
- 正式比较时建议改为 `NORMALIZATION="capacity"`，并填写真实装机容量；`p95` 会隐藏一部分场站间的真实纵向差异。
- 这个 Notebook 用全年/所选数据范围的中位数曲线估计场站映射。若目标站数据参与映射估计，这属于无标签目标域自适应，而不是严格零样本 OOD。